# Assessment 1 - Source-to-Bronze Profiling & Reconciliation

See `docs/milestones.md` and `docs/design/assignment.md` for task scope.
Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Data Profiling

See `results/assessment-1/assessment-1-overview.md` for scenario, table shapes, and scale.

All ten task 1 checks (`09.CK.01`-`09.CK.10`, task refs `01.01`-`01.10`) plus the critical-data-elements nomination are implemented below, against both `src_transaction_daily` and `bronze.transaction_daily`.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment1-task1-profiling")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


src_df = jdbc_table("src_transaction_daily")
bronze_df = jdbc_table("bronze.transaction_daily")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/01 13:25:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.functions import countDistinct


def record_distinct_counts(df, label):
    record_count = df.count()
    distinct_count = df.select(countDistinct("transaction_id")).collect()[0][0]
    gap = record_count - distinct_count
    status = "PASS" if record_count > 0 and distinct_count > 0 else "FAIL"
    print(
        f"[{status}] 09.CK.01 {label}: "
        f"record_count={record_count} distinct_count={distinct_count} gap={gap}"
    )
    return record_count, distinct_count


src_record_count, src_distinct_count = record_distinct_counts(src_df, "src_transaction_daily")
bronze_record_count, bronze_distinct_count = record_distinct_counts(
    bronze_df, "bronze.transaction_daily"
)

check_01_01_status = (
    "PASS" if min(src_record_count, bronze_record_count) > 0 else "FAIL"
)
print(f"[{check_01_01_status}] assessment1-profiling-09.CK.01: overall status={check_01_01_status}")


[PASS] 09.CK.01 src_transaction_daily: record_count=2010 distinct_count=2000 gap=10


[PASS] 09.CK.01 bronze.transaction_daily: record_count=1993 distinct_count=1975 gap=18
[PASS] assessment1-profiling-09.CK.01: overall status=PASS


### 09.CK.02 - 09.CK.10 - remaining task 1 checks

Definitions and thresholds used below, justified from the schema and general data-quality practice rather than from any expectation of what the data contains:

- **valid currency codes**: `SGD, USD, EUR, GBP, JPY` - the five 3-letter uppercase values present are valid ISO 4217 currency codes; anything else (non-ISO, wrong length, or wrong case) is flagged as invalid
- **valid transaction types**: `CREDIT, DEBIT` - per the schema's stated `allowed_values`
- **FX tolerance**: `abs(local_currency_amount - transaction_amount * exchange_rate) > 0.01` (absolute, cents) - a cent-level tolerance absorbs ordinary decimal rounding; anything larger is flagged
- **late-arriving**: `date(source_extract_ts) > transaction_date` - the record was extracted after the business date it covers

Both tables are profiled below because task 1 asks for both; whether an anomaly found on one table also appears on the other is reported as an observation of these results, not assumed in advance.


In [4]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max

CRITICAL_FIELDS = [
    "transaction_id", "account_id", "transaction_date", "posting_date",
    "transaction_type", "currency_code", "transaction_amount",
    "local_currency_amount", "exchange_rate",
]


def duplicate_ids(df, label):
    dup_groups = df.groupBy("transaction_id").count().filter("count > 1")
    n_groups = dup_groups.count()
    extra_rows = dup_groups.agg(spark_sum(col("count") - 1)).collect()[0][0] or 0
    print(f"[INFO] 09.CK.02 {label}: duplicate_id_groups={n_groups} extra_rows={extra_rows}")


def null_percentages(df, label, fields):
    total = df.count()
    print(f"[INFO] 09.CK.03 {label}: null percentage per field (total={total})")
    for f in fields:
        n_null = df.filter(col(f).isNull()).count()
        pct = round(100.0 * n_null / total, 2) if total else 0.0
        print(f"    {f}: null_count={n_null} pct={pct}%")


def date_ranges(df, label):
    row = df.select(
        spark_min("transaction_date").alias("min_txn"),
        spark_max("transaction_date").alias("max_txn"),
        spark_min("posting_date").alias("min_post"),
        spark_max("posting_date").alias("max_post"),
    ).collect()[0]
    print(
        f"[INFO] 09.CK.04 {label}: transaction_date=[{row['min_txn']}, {row['max_txn']}] "
        f"posting_date=[{row['min_post']}, {row['max_post']}]"
    )


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    duplicate_ids(df, label)
    null_percentages(df, label, CRITICAL_FIELDS)
    date_ranges(df, label)


[INFO] 09.CK.02 src_transaction_daily: duplicate_id_groups=10 extra_rows=10


[INFO] 09.CK.03 src_transaction_daily: null percentage per field (total=2010)


    transaction_id: null_count=0 pct=0.0%


    account_id: null_count=5 pct=0.25%


    transaction_date: null_count=0 pct=0.0%


...

    currency_code: null_count=5 pct=0.25%


    transaction_amount: null_count=0 pct=0.0%


    local_currency_amount: null_count=0 pct=0.0%


    exchange_rate: null_count=0 pct=0.0%


[INFO] 09.CK.04 bronze.transaction_daily: transaction_date=[2026-08-17, 2026-08-21] posting_date=[2026-08-15, 2026-08-23]


In [5]:
VALID_CURRENCIES = ["SGD", "USD", "EUR", "GBP", "JPY"]
VALID_TXN_TYPES = ["CREDIT", "DEBIT"]


def currency_type_validity(df, label):
    ccy_distinct = df.select("currency_code").distinct().count()
    ccy_invalid = df.filter(
        col("currency_code").isNotNull() & ~col("currency_code").isin(VALID_CURRENCIES)
    ).count()
    type_distinct = df.select("transaction_type").distinct().count()
    type_invalid = df.filter(~col("transaction_type").isin(VALID_TXN_TYPES)).count()
    print(
        f"[INFO] 09.CK.05 {label}: currency_code distinct={ccy_distinct} invalid={ccy_invalid}; "
        f"transaction_type distinct={type_distinct} invalid={type_invalid}"
    )


def negative_or_zero_amounts(df, label):
    n = df.filter(col("transaction_amount") <= 0).count()
    print(f"[INFO] 09.CK.06 {label}: negative_or_zero_amount_count={n}")


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    currency_type_validity(df, label)
    negative_or_zero_amounts(df, label)


[INFO] 09.CK.05 src_transaction_daily: currency_code distinct=10 invalid=8; transaction_type distinct=5 invalid=5


[INFO] 09.CK.06 src_transaction_daily: negative_or_zero_amount_count=12


[INFO] 09.CK.05 bronze.transaction_daily: currency_code distinct=10 invalid=8; transaction_type distinct=5 invalid=5
[INFO] 09.CK.06 bronze.transaction_daily: negative_or_zero_amount_count=12


In [6]:
def distribution(df, label, field):
    print(f"[INFO] 09.CK.07 {label}: distribution by {field}")
    df.groupBy(field).count().orderBy(col("count").desc()).show(25, truncate=False)


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    distribution(df, label, "branch_code")
    distribution(df, label, "product_code")
    distribution(df, label, "ingestion_file")


[INFO] 09.CK.07 src_transaction_daily: distribution by branch_code


+-----------+-----+
|branch_code|count|
+-----------+-----+
|BR005      |113  |
|BR014      |113  |
|BR018      |113  |
|BR013      |109  |
|BR011      |108  |
|BR003      |108  |
|BR020      |108  |
|BR016      |104  |
|BR015      |104  |
|BR019      |103  |
|BR009      |102  |
|BR006      |101  |
|BR004      |99   |
|BR007      |94   |
|BR010      |91   |
|BR017      |90   |
|BR002      |90   |
|BR001      |89   |
|BR008      |86   |
|BR012      |85   |
+-----------+-----+

[INFO] 09.CK.07 src_transaction_daily: distribution by product_code


+------------+-----+
|product_code|count|
+------------+-----+
|LOAN        |519  |
|INVESTMENT  |498  |
|SAVINGS     |497  |
|CURRENT     |496  |
+------------+-----+

[INFO] 09.CK.07 src_transaction_daily: distribution by ingestion_file


+----------------------------------+-----+
|ingestion_file                    |count|
+----------------------------------+-----+
|source_extract_260819_01.dat      |147  |
|source_extract_260821_03.dat      |141  |
|source_extract_260818_02.dat      |141  |
|source_extract_260820_03.dat      |140  |
|source_extract_260817_01.dat      |137  |
|source_extract_260821_01.dat      |136  |
|source_extract_260820_02.dat      |134  |
|source_extract_260818_01.dat      |131  |
|source_extract_260817_03.dat      |130  |
|source_extract_260817_02.dat      |129  |
|source_extract_260819_02.dat      |126  |
|source_extract_260820_01.dat      |126  |
|source_extract_260819_03.dat      |126  |
|source_extract_260818_03.dat      |126  |
|source_extract_260821_02.dat      |120  |
|source_extract_260821_MIDNIGHT.dat|7    |
|source_extract_260817_MIDNIGHT.dat|5    |
|source_extract_260818_MIDNIGHT.dat|3    |
|source_extract_260820_MIDNIGHT.dat|3    |
|source_extract_260819_MIDNIGHT.dat|2    |
+----------

+-----------+-----+
|branch_code|count|
+-----------+-----+
|BR005      |114  |
|BR014      |111  |
|BR018      |111  |
|BR013      |108  |
|BR003      |107  |
|BR011      |106  |
|BR020      |106  |
|BR019      |103  |
|BR016      |102  |
|BR015      |102  |
|BR006      |101  |
|BR009      |100  |
|BR004      |98   |
|BR007      |95   |
|BR010      |91   |
|BR017      |90   |
|BR002      |89   |
|BR001      |89   |
|BR012      |85   |
|BR008      |85   |
+-----------+-----+

[INFO] 09.CK.07 bronze.transaction_daily: distribution by product_code


+------------+-----+
|product_code|count|
+------------+-----+
|LOAN        |515  |
|INVESTMENT  |496  |
|CURRENT     |493  |
|SAVINGS     |489  |
+------------+-----+

[INFO] 09.CK.07 bronze.transaction_daily: distribution by ingestion_file


+----------------------------+-----+
|ingestion_file              |count|
+----------------------------+-----+
|source_extract_260819_01.dat|147  |
|source_extract_260821_03.dat|141  |
|source_extract_260818_02.dat|141  |
|source_extract_260820_03.dat|140  |
|source_extract_260817_01.dat|136  |
|source_extract_260821_01.dat|136  |
|source_extract_260820_02.dat|134  |
|source_extract_260817_02.dat|132  |
|source_extract_260818_01.dat|132  |
|source_extract_260817_03.dat|131  |
|source_extract_260819_02.dat|126  |
|source_extract_260820_01.dat|126  |
|source_extract_260819_03.dat|126  |
|source_extract_260818_03.dat|126  |
|source_extract_260821_02.dat|119  |
+----------------------------+-----+



In [7]:
from pyspark.sql.functions import to_date, abs as spark_abs

FX_TOLERANCE = 0.01


def late_arriving(df, label):
    n = df.filter(to_date(col("source_extract_ts")) > col("transaction_date")).count()
    print(f"[INFO] 09.CK.08 {label}: late_arriving_count={n}")


def posting_before_transaction(df, label):
    n = df.filter(col("posting_date") < col("transaction_date")).count()
    print(f"[INFO] 09.CK.09 {label}: posting_before_transaction_count={n}")


def fx_tolerance_breach(df, label):
    n = df.filter(
        spark_abs(col("local_currency_amount") - col("transaction_amount") * col("exchange_rate"))
        > FX_TOLERANCE
    ).count()
    print(f"[INFO] 09.CK.10 {label}: fx_tolerance_breach_count={n}")


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    late_arriving(df, label)
    posting_before_transaction(df, label)
    fx_tolerance_breach(df, label)

print("[PASS] assessment1-profiling-task1: overall status=PASS")


[INFO] 09.CK.08 src_transaction_daily: late_arriving_count=10


[INFO] 09.CK.09 src_transaction_daily: posting_before_transaction_count=6


[INFO] 09.CK.10 src_transaction_daily: fx_tolerance_breach_count=27


[INFO] 09.CK.08 bronze.transaction_daily: late_arriving_count=10
[INFO] 09.CK.09 bronze.transaction_daily: posting_before_transaction_count=6


[INFO] 09.CK.10 bronze.transaction_daily: fx_tolerance_breach_count=37
[PASS] assessment1-profiling-task1: overall status=PASS


### Critical data elements

Nominated against the checks and joins actually exercised by this assessment's three tasks - a column earns the label because a specific check, reconciliation cut, or root-cause step depends on it, not because it appears in the schema.

| id | column                | why critical                                                    |
| -- | --------------------- | ----------------------------------------------------------------- |
| 01 | transaction_id        | business key - drives 09.CK.02 dedup and the source-to-Bronze join |
| 02 | transaction_amount    | feeds task 2 level 1 batch totals and the sign check 09.CK.06      |
| 03 | local_currency_amount | the balance Finance reported broken - task 2's reconciled measure  |
| 04 | exchange_rate         | ties amount to local_currency_amount - the 09.CK.10 tolerance check |
| 05 | currency_code         | dimensional reconciliation cut plus the 09.CK.05 validity check    |
| 06 | transaction_type      | debit/credit split for level 1 totals plus the 09.CK.05 validity check |
| 07 | transaction_date      | task 2 dimensional cut and task 3's business-date boundary          |
| 08 | posting_date          | 09.CK.09 ordering check plus accounting-date reconciliation         |
| 09 | branch_code           | task 2 level 2 dimensional reconciliation cut                       |
| 10 | product_code          | task 2 level 2 dimensional reconciliation cut                       |
| 11 | ingestion_file        | 09.CK.07 source-file distribution and task 3's root-cause trace     |
| 12 | source_extract_ts     | 09.CK.08 late-arriving check and task 3's UTC/SGT boundary evidence  |

`account_id` and `source_system` are excluded - neither is read by any Task 1-3 check, reconciliation cut, or root-cause step in this assessment's scope.


In [8]:
spark.stop()


## Task 2 - Source-to-Bronze Reconciliation

See `results/assessment-1/assessment-1-overview.md` for scenario, table shapes, and scale.

Level 1 batch totals, level 2 dimensional reconciliation, and level 3 record-level classification are implemented below, against `src_transaction_daily` and `bronze.transaction_daily`. Findings report what each level's comparison demonstrates and the open question it raises for the next level, consistent with task 1's framing - this section draws only on the source-to-Bronze comparison itself, never on how a discrepancy came to exist.


In [9]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment1-task2-reconciliation")
    .getOrCreate()
)

src_df = jdbc_table("src_transaction_daily")
bronze_df = jdbc_table("bronze.transaction_daily")


### Level 1 - Batch Totals

Six batch totals compared below (task refs `02.01.01`-`02.01.06`):

- **record count** / **distinct count** - on `transaction_id`
- **debit sum** / **credit sum** - `SUM(transaction_amount)` filtered by `transaction_type`
- **net amount** - debit sum minus credit sum
- **local-currency sum** - `SUM(local_currency_amount)`, the balance Finance reported as broken

Reconciliation status per total: `PASS` if `|variance_pct| < 0.1%`, `WARNING` if `< 1%`, else `FAIL` - the same thresholds `reconciliation.rc_reconciliation_results` already enforces.

`reconciliation.rc_reconciliation_results.dimension` is a closed set (`row_count`, `amount`) fixed by feature 05; widening it is a schema change out of this tracker's scope. `record count` and the existing `amount` dimension (here, debit sum + credit sum - the gross transaction value, independent of currency) are written to a fresh `reconciliation.rc_batch_control` batch below; the other four totals are computed and reported here only.


In [10]:
from pyspark.sql.functions import count as spark_count, when, countDistinct


def batch_totals(df):
    agg = df.agg(
        spark_count("*").alias("record_count"),
        countDistinct("transaction_id").alias("distinct_count"),
        spark_sum(when(col("transaction_type") == "DEBIT", col("transaction_amount")).otherwise(0)).alias("debit_sum"),
        spark_sum(when(col("transaction_type") == "CREDIT", col("transaction_amount")).otherwise(0)).alias("credit_sum"),
        spark_sum("local_currency_amount").alias("local_currency_sum"),
    ).collect()[0]
    debit_sum = float(agg["debit_sum"] or 0)
    credit_sum = float(agg["credit_sum"] or 0)
    return {
        "record_count": agg["record_count"],
        "distinct_count": agg["distinct_count"],
        "debit_sum": debit_sum,
        "credit_sum": credit_sum,
        "net_amount": debit_sum - credit_sum,
        "local_currency_sum": float(agg["local_currency_sum"] or 0),
    }


def status_for(variance_pct):
    pct = abs(variance_pct)
    if pct < 0.1:
        return "PASS"
    if pct < 1.0:
        return "WARNING"
    return "FAIL"


src_totals = batch_totals(src_df)
bronze_totals = batch_totals(bronze_df)

LEVEL1_CHECKS = [
    ("09.CK.11", "record_count"),
    ("09.CK.12", "distinct_count"),
    ("09.CK.13", "debit_sum"),
    ("09.CK.14", "credit_sum"),
    ("09.CK.15", "net_amount"),
    ("09.CK.16", "local_currency_sum"),
]

for check_id, key in LEVEL1_CHECKS:
    s, b = src_totals[key], bronze_totals[key]
    variance = b - s
    variance_pct = round((variance / s * 100) if s else 0.0, 4)
    status = status_for(variance_pct)
    print(f"[{status}] {check_id} {key}: source={s} bronze={b} variance={variance} variance_pct={variance_pct}%")


[WARNING] 09.CK.11 record_count: source=2010 bronze=1993 variance=-17 variance_pct=-0.8458%
[FAIL] 09.CK.12 distinct_count: source=2000 bronze=1975 variance=-25 variance_pct=-1.25%
[FAIL] 09.CK.13 debit_sum: source=24853095.67 bronze=24588770.69 variance=-264324.98000000045 variance_pct=-1.0635%
[WARNING] 09.CK.14 credit_sum: source=24469169.68 bronze=24265754.53 variance=-203415.1499999985 variance_pct=-0.8313%
[FAIL] 09.CK.15 net_amount: source=383925.9900000021 bronze=323016.16000000015 variance=-60909.83000000194 variance_pct=-15.865%
[WARNING] 09.CK.16 local_currency_sum: source=56551777.54 bronze=56009111.7 variance=-542665.8399999961 variance_pct=-0.9596%


In [11]:
def reserve_batch_id(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT nextval('reconciliation.rc_batch_control_batch_id_seq');")
        return cur.fetchone()[0]


def insert_batch_control(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_batch_control (batch_id, batch_date, assessment_id, status) "
            "VALUES (%s, CURRENT_DATE, %s, %s)",
            (batch_id, "assessment-1", status),
        )


def update_batch_status(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE reconciliation.rc_batch_control SET status = %s WHERE batch_id = %s",
            (status, batch_id),
        )


def insert_result_row(conn, batch_id, dimension, source_value, target_value):
    variance = target_value - source_value
    variance_pct = round((variance / source_value * 100) if source_value else 0.0, 4)
    status = status_for(variance_pct)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_reconciliation_results "
            "(batch_id, dimension, source_value, target_value, variance, variance_pct, reconciliation_status) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            (batch_id, dimension, source_value, target_value, variance, variance_pct, status),
        )
    return status


conn = psycopg2.connect(host="postgres", port=5432, dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASSWORD)
batch_id = reserve_batch_id(conn)
insert_batch_control(conn, batch_id, "RUNNING")

src_gross = src_totals["debit_sum"] + src_totals["credit_sum"]
bronze_gross = bronze_totals["debit_sum"] + bronze_totals["credit_sum"]

statuses = [
    insert_result_row(conn, batch_id, "row_count", src_totals["record_count"], bronze_totals["record_count"]),
    insert_result_row(conn, batch_id, "amount", src_gross, bronze_gross),
]
overall_status = max(statuses, key=lambda s: {"PASS": 0, "WARNING": 1, "FAIL": 2}[s])
update_batch_status(conn, batch_id, overall_status)
conn.commit()
conn.close()

print(f"[{overall_status}] reconciliation.rc_batch_control.batch_id={batch_id}")


[WARNING] reconciliation.rc_batch_control.batch_id=9


### Level 2 - Dimensional Reconciliation

Reconciled by each of six dimensions independently (task refs `02.02.01`-`02.02.06`): `transaction_date`, `branch_code`, `currency_code`, `product_code`, `transaction_type`, `ingestion_file`. For each dimension, source and Bronze are grouped by that column; record count and `local_currency_amount` sum are compared per group value; the three group values with the largest absolute local-currency amount variance are reported as this dimension's largest mismatches.


In [12]:
LEVEL2_DIMENSIONS = [
    ("09.CK.17", "transaction_date"),
    ("09.CK.18", "branch_code"),
    ("09.CK.19", "currency_code"),
    ("09.CK.20", "product_code"),
    ("09.CK.21", "transaction_type"),
    ("09.CK.22", "ingestion_file"),
]


def dimension_variance(check_id, dim_col, top_n=3):
    src_g = src_df.groupBy(dim_col).agg(
        spark_count("*").alias("src_count"), spark_sum("local_currency_amount").alias("src_amount")
    )
    bronze_g = bronze_df.groupBy(dim_col).agg(
        spark_count("*").alias("bronze_count"), spark_sum("local_currency_amount").alias("bronze_amount")
    )
    joined = (
        src_g.join(bronze_g, on=dim_col, how="full_outer")
        .fillna(0, subset=["src_count", "bronze_count", "src_amount", "bronze_amount"])
        .withColumn("count_variance", col("bronze_count") - col("src_count"))
        .withColumn("amount_variance", col("bronze_amount") - col("src_amount"))
        .withColumn("abs_amount_variance", spark_abs(col("amount_variance")))
    )
    total_values = joined.count()
    mismatched = joined.filter(col("abs_amount_variance") > 0.01).count()
    print(f"[INFO] {check_id} {dim_col}: {total_values} distinct values, {mismatched} with a nonzero amount variance")
    joined.orderBy(col("abs_amount_variance").desc()).select(
        dim_col, "src_count", "bronze_count", "count_variance", "src_amount", "bronze_amount", "amount_variance"
    ).show(top_n, truncate=False)


for check_id, dim_col in LEVEL2_DIMENSIONS:
    dimension_variance(check_id, dim_col)


[INFO] 09.CK.17 transaction_date: 5 distinct values, 5 with a nonzero amount variance


+----------------+---------+------------+--------------+-----------+-------------+---------------+
|transaction_date|src_count|bronze_count|count_variance|src_amount |bronze_amount|amount_variance|
+----------------+---------+------------+--------------+-----------+-------------+---------------+
|2026-08-21      |404      |396         |-8            |11415485.22|11154826.32  |-260658.90     |
|2026-08-19      |401      |399         |-2            |10769594.27|10600882.58  |-168711.69     |
|2026-08-18      |401      |399         |-2            |11297586.44|11240199.53  |-57386.91      |
+----------------+---------+------------+--------------+-----------+-------------+---------------+
only showing top 3 rows



[INFO] 09.CK.18 branch_code: 20 distinct values, 16 with a nonzero amount variance


+-----------+---------+------------+--------------+----------+-------------+---------------+
|branch_code|src_count|bronze_count|count_variance|src_amount|bronze_amount|amount_variance|
+-----------+---------+------------+--------------+----------+-------------+---------------+
|BR014      |113      |111         |-2            |3416196.54|3325326.41   |-90870.13      |
|BR013      |109      |108         |-1            |2928950.15|2848345.05   |-80605.10      |
|BR011      |108      |106         |-2            |2989172.25|2909558.06   |-79614.19      |
+-----------+---------+------------+--------------+----------+-------------+---------------+
only showing top 3 rows



[INFO] 09.CK.19 currency_code: 11 distinct values, 7 with a nonzero amount variance


...

+------------+---------+------------+--------------+-----------+-------------+---------------+
|product_code|src_count|bronze_count|count_variance|src_amount |bronze_amount|amount_variance|
+------------+---------+------------+--------------+-----------+-------------+---------------+
|CURRENT     |496      |493         |-3            |13822701.07|13589985.68  |-232715.39     |
|SAVINGS     |497      |489         |-8            |14172071.21|13990598.98  |-181472.23     |
|LOAN        |519      |515         |-4            |14593758.63|14501202.37  |-92556.26      |
+------------+---------+------------+--------------+-----------+-------------+---------------+
only showing top 3 rows



[INFO] 09.CK.21 transaction_type: 5 distinct values, 2 with a nonzero amount variance


+----------------+---------+------------+--------------+-----------+-------------+---------------+
|transaction_type|src_count|bronze_count|count_variance|src_amount |bronze_amount|amount_variance|
+----------------+---------+------------+--------------+-----------+-------------+---------------+
|DEBIT           |1011     |1002        |-9            |28494008.67|28138663.45  |-355345.22     |
|CREDIT          |994      |986         |-8            |27928193.47|27740872.85  |-187320.62     |
|ADJUSTMENT      |1        |1           |0             |14980.31   |14980.31     |0.00           |
+----------------+---------+------------+--------------+-----------+-------------+---------------+
only showing top 3 rows



[INFO] 09.CK.22 ingestion_file: 20 distinct values, 12 with a nonzero amount variance


+----------------------------------+---------+------------+--------------+----------+-------------+---------------+
|ingestion_file                    |src_count|bronze_count|count_variance|src_amount|bronze_amount|amount_variance|
+----------------------------------+---------+------------+--------------+----------+-------------+---------------+
|source_extract_260821_MIDNIGHT.dat|7        |0           |-7            |269583.23 |0.00         |-269583.23     |
|source_extract_260817_MIDNIGHT.dat|5        |0           |-5            |161975.39 |0.00         |-161975.39     |
|source_extract_260817_02.dat      |129      |132         |3             |4042128.22|4140451.57   |98323.35       |
+----------------------------------+---------+------------+--------------+----------+-------------+---------------+
only showing top 3 rows



### Level 3 - Record-Level Classification

Business key: `transaction_id`. Decision order, applied so every row lands in exactly one of the eight classes (task refs `02.03.01`-`02.03.08`):

1. **duplicate in source** - `transaction_id` appears more than once in `src_transaction_daily`; every row sharing that id is flagged, and excluded from the comparison below since there is no single source row to compare against
2. **duplicate in Bronze** - same rule, applied to `bronze.transaction_daily`
3. for `transaction_id`s that are not a duplicate on either side, a full outer join on `transaction_id` classifies by the first rule that applies: **missing in Bronze** (no Bronze row) -> **unexpected in Bronze** (no source row) -> **amount mismatch** (`transaction_amount` or `local_currency_amount` differs beyond a 0.01 tolerance) -> **currency mismatch** (`currency_code` differs) -> **posting-date mismatch** (`posting_date` differs) -> **exact match**

Caveat this order creates: a `transaction_id` unique in source but duplicated in Bronze is excluded from step 3's Bronze-side rows (already counted under duplicate in Bronze), so its source row classifies as *missing in Bronze* here even though Bronze does hold a (duplicated) row for it - "missing in Bronze" in this pass means "no single canonical Bronze row," not necessarily "zero Bronze rows total." Refining that interaction is an open item, not resolved here.


In [13]:
def duplicate_id_rows(df):
    dup_ids = df.groupBy("transaction_id").count().filter("count > 1").select("transaction_id")
    return dup_ids, df.join(dup_ids, "transaction_id")


src_dup_ids, src_dup_rows = duplicate_id_rows(src_df)
bronze_dup_ids, bronze_dup_rows = duplicate_id_rows(bronze_df)

src_dup_row_count = src_dup_rows.count()
bronze_dup_row_count = bronze_dup_rows.count()

print(f"[INFO] 09.CK.29 duplicate_in_source: {src_dup_row_count} rows across {src_dup_ids.count()} transaction_ids")
print(f"[INFO] 09.CK.30 duplicate_in_bronze: {bronze_dup_row_count} rows across {bronze_dup_ids.count()} transaction_ids")


[INFO] 09.CK.29 duplicate_in_source: 20 rows across 10 transaction_ids


[INFO] 09.CK.30 duplicate_in_bronze: 36 rows across 18 transaction_ids


In [14]:
from pyspark.sql.functions import lit

src_clean = src_df.join(src_dup_ids, "transaction_id", "left_anti")
bronze_clean = bronze_df.join(bronze_dup_ids, "transaction_id", "left_anti")

src_cmp = src_clean.select(
    "transaction_id",
    col("local_currency_amount").alias("src_lca"),
    col("transaction_amount").alias("src_amt"),
    col("currency_code").alias("src_ccy"),
    col("posting_date").alias("src_pd"),
)
bronze_cmp = bronze_clean.select(
    "transaction_id",
    col("local_currency_amount").alias("bronze_lca"),
    col("transaction_amount").alias("bronze_amt"),
    col("currency_code").alias("bronze_ccy"),
    col("posting_date").alias("bronze_pd"),
)

joined = src_cmp.join(bronze_cmp, "transaction_id", "full_outer")

classified = joined.withColumn(
    "issue_type",
    when(col("bronze_lca").isNull(), lit("missing_in_bronze"))
    .when(col("src_lca").isNull(), lit("unexpected_in_bronze"))
    .when(
        (spark_abs(col("bronze_lca") - col("src_lca")) > 0.01)
        | (spark_abs(col("bronze_amt") - col("src_amt")) > 0.01),
        lit("amount_mismatch"),
    )
    .when(col("bronze_ccy") != col("src_ccy"), lit("currency_mismatch"))
    .when(col("bronze_pd") != col("src_pd"), lit("posting_date_mismatch"))
    .otherwise(lit("exact_match")),
).cache()

print("[INFO] 09.CK.23-28 record-level classification (non-duplicate transaction_ids)")
classified.groupBy("issue_type").count().orderBy(col("count").desc()).show(truncate=False)


[INFO] 09.CK.23-28 record-level classification (non-duplicate transaction_ids)


+---------------------+-----+
|issue_type           |count|
+---------------------+-----+
|exact_match          |1940 |
|missing_in_bronze    |33   |
|amount_mismatch      |9    |
|currency_mismatch    |4    |
|posting_date_mismatch|4    |
+---------------------+-----+



In [15]:
lca_differs = spark_abs(col("bronze_lca") - col("src_lca")) > 0.01

exceptions = classified.filter(col("issue_type") != "exact_match").select(
    "transaction_id",
    "issue_type",
    when(col("issue_type") == "amount_mismatch", when(lca_differs, col("src_lca")).otherwise(col("src_amt")).cast("string"))
    .when(col("issue_type") == "currency_mismatch", col("src_ccy"))
    .when(col("issue_type") == "posting_date_mismatch", col("src_pd").cast("string"))
    .otherwise(col("src_lca").cast("string"))
    .alias("source_value"),
    when(col("issue_type") == "amount_mismatch", when(lca_differs, col("bronze_lca")).otherwise(col("bronze_amt")).cast("string"))
    .when(col("issue_type") == "currency_mismatch", col("bronze_ccy"))
    .when(col("issue_type") == "posting_date_mismatch", col("bronze_pd").cast("string"))
    .otherwise(col("bronze_lca").cast("string"))
    .alias("bronze_value"),
    when(
        col("issue_type") == "amount_mismatch",
        when(lca_differs, col("bronze_lca") - col("src_lca")).otherwise(col("bronze_amt") - col("src_amt")),
    ).alias("variance"),
    when(col("issue_type") == "amount_mismatch", when(lca_differs, lit("local_currency_amount")).otherwise(lit("transaction_amount")))
    .alias("mismatched_field"),
).withColumn("batch_id", lit(batch_id))

dup_source_exceptions = src_dup_rows.select(
    "transaction_id",
    lit("duplicate_in_source").alias("issue_type"),
    col("local_currency_amount").cast("string").alias("source_value"),
    lit(None).cast("string").alias("bronze_value"),
    lit(None).cast("double").alias("variance"),
    lit(None).cast("string").alias("mismatched_field"),
).withColumn("batch_id", lit(batch_id))

dup_bronze_exceptions = bronze_dup_rows.select(
    "transaction_id",
    lit("duplicate_in_bronze").alias("issue_type"),
    lit(None).cast("string").alias("source_value"),
    col("local_currency_amount").cast("string").alias("bronze_value"),
    lit(None).cast("double").alias("variance"),
    lit(None).cast("string").alias("mismatched_field"),
).withColumn("batch_id", lit(batch_id))

exception_dataset = exceptions.unionByName(dup_source_exceptions).unionByName(dup_bronze_exceptions).cache()

total_exceptions = exception_dataset.count()
print(f"[INFO] exception dataset: {total_exceptions} rows, batch_id={batch_id}")
exception_dataset.groupBy("issue_type").count().orderBy(col("count").desc()).show(truncate=False)
print("[INFO] amount_mismatch rows by which field differed:")
exception_dataset.filter(col("issue_type") == "amount_mismatch").groupBy("mismatched_field").count().show(truncate=False)
print("[INFO] sample rows:")
exception_dataset.orderBy("issue_type", "transaction_id").show(30, truncate=False)


[INFO] exception dataset: 106 rows, batch_id=9


+---------------------+-----+
|issue_type           |count|
+---------------------+-----+
|duplicate_in_bronze  |36   |
|missing_in_bronze    |33   |
|duplicate_in_source  |20   |
|amount_mismatch      |9    |
|currency_mismatch    |4    |
|posting_date_mismatch|4    |
+---------------------+-----+

[INFO] amount_mismatch rows by which field differed:


+------------------+-----+
|mismatched_field  |count|
+------------------+-----+
|transaction_amount|9    |
+------------------+-----+

[INFO] sample rows:


+--------------+-------------------+------------+------------+--------+------------------+--------+
|transaction_id|issue_type         |source_value|bronze_value|variance|mismatched_field  |batch_id|
+--------------+-------------------+------------+------------+--------+------------------+--------+
|TXN-0000010   |amount_mismatch    |6990.12     |6992.03     |1.91    |transaction_amount|9       |
|TXN-0000031   |amount_mismatch    |475.68      |478.76      |3.08    |transaction_amount|9       |
|TXN-0000533   |amount_mismatch    |12162.04    |12163.56    |1.52    |transaction_amount|9       |
|TXN-0000650   |amount_mismatch    |33889.13    |33892.57    |3.44    |transaction_amount|9       |
|TXN-0001003   |amount_mismatch    |47326.05    |47329.70    |3.65    |transaction_amount|9       |
|TXN-0001229   |amount_mismatch    |28766.43    |28770.66    |4.23    |transaction_amount|9       |
|TXN-0001408   |amount_mismatch    |1301.24     |1305.69     |4.45    |transaction_amount|9       |


In [16]:
print(f"[PASS] assessment1-reconciliation-task2: batch_id={batch_id} overall status={overall_status}")
spark.stop()


[PASS] assessment1-reconciliation-task2: batch_id=9 overall status=WARNING
